In [1]:
import numpy as np
import pandas as pd
import os
import h5py
import json
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# LNN
import torch
import torch.nn as nn
from LiqudNet.liquidnet.main import LiquidNet
from torch.utils.data import random_split, DataLoader

#KAN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, TensorDataset
from tqdm.auto import tqdm
import numpy as np

import torch
import torch.nn.functional as F
import math

from torchsummary import summary

# Preprocessing
import re
import nltk
import spacy
import fasttext
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [2]:
class FakeNewsDataset(Dataset):
    def __init__(self, word_embeddings_file, labels_file = None):
#         self.word_embeddings = pd.read_csv(word_embeddings_file).values
        self.word_embeddings = np.load(word_embeddings_file)

        # Load labels if available
        if labels_file:
            self.labels = pd.read_csv(labels_file, header = None)[0].values
        else:
            self.labels = None

    def __len__(self):
        return len(self.word_embeddings)  # Assuming word embeddings match character embeddings

    def __getitem__(self, idx):
        word_embedding = torch.tensor(self.word_embeddings[idx], dtype = torch.float32)

        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype = torch.long)
            return word_embedding, label
        else:
            return word_embedding

In [3]:
# Define dataset sizes
def split_dataset(dataset, train_ratio = 0.8, val_ratio = 0.1):
    """
    Split the dataset into train, validation, and test subsets.
    :param dataset: The complete dataset.
    :param train_ratio: Proportion of the data to be used for training.
    :param val_ratio: Proportion of the data to be used for validation.
    :return: Train, validation, and test subsets.
    """
    total_size = len(dataset)
    train_size = int(train_ratio * total_size)
    val_size = int(val_ratio * total_size)
    test_size = total_size - train_size - val_size

    return random_split(dataset, [train_size, val_size, test_size])

In [4]:
# Path to the embeddings and labels
word_embeddings_file = '../Datasets/Final/Fixed_Embeddings_Sequence.npy'
labels_file = '../Datasets/Final/Encoded_Labels.csv'  # CSV file with a 'type' column for real/fake labels
# Initialize the dataset with both embeddings and labels
dataset = FakeNewsDataset(word_embeddings_file, labels_file)

In [5]:
train_dataset, val_dataset, test_dataset = split_dataset(dataset, train_ratio = 0.8, val_ratio = 0.1)

In [6]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, num_workers = 0)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False, num_workers = 0)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False, num_workers = 0)

## Model

In [7]:
class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size = 5,
        spline_order = 3,
        scale_noise = 0.1,
        scale_base = 1.0,
        scale_spline = 1.0,
        enable_standalone_scale_spline = True,
        base_activation = torch.nn.SiLU,
        grid_eps = 0.02,
        grid_range = [-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a = math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                # torch.nn.init.constant_(self.spline_scaler, self.scale_spline)
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a = math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        Compute the B-spline bases for the given input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).

        Returns:
            torch.Tensor: B-spline bases tensor of shape (batch_size, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )  # (in_features, grid_size + 2 * spline_order + 1)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        assert x.size(-1) == self.in_features
        original_shape = x.shape
        x = x.reshape(-1, self.in_features)

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        output = base_output + spline_output

        output = output.reshape(*original_shape[:-1], self.out_features)
        return output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin = 0.01):
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  # (batch, in, coeff)
        splines = splines.permute(1, 0, 2)  # (in, batch, coeff)
        orig_coeff = self.scaled_spline_weight  # (out, in, coeff)
        orig_coeff = orig_coeff.permute(1, 2, 0)  # (in, coeff, out)
        unreduced_spline_output = torch.bmm(splines, orig_coeff)  # (in, batch, out)
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        )  # (batch, in, out)

        # sort each channel individually to collect data distribution
        x_sorted = torch.sort(x, dim = 0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device = x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype = torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device = x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device = x.device).unsqueeze(1),
            ],
            dim = 0,
        )

        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """
        Compute the regularization loss.

        This is a dumb simulation of the original L1 regularization as stated in the
        paper, since the original one requires computing absolutes and entropy from the
        expanded (batch, in_features, out_features) intermediate tensor, which is hidden
        behind the F.linear function if we want an memory efficient implementation.

        The L1 regularization is now computed as mean absolute value of the spline
        weights. The authors implementation also includes this term in addition to the
        sample-based regularization.
        """
        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )

In [8]:
class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size = 5,
        spline_order = 3,
        scale_noise = 0.1,
        scale_base = 1.0,
        scale_spline = 1.0,
        base_activation = torch.nn.SiLU,
        grid_eps = 0.02,
        grid_range = [-1, 1],
    ):
        super(KAN, self).__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size = grid_size,
                    spline_order = spline_order,
                    scale_noise = scale_noise,
                    scale_base = scale_base,
                    scale_spline = scale_spline,
                    base_activation = base_activation,
                    grid_eps = grid_eps,
                    grid_range = grid_range,
                )
            )

    def forward(self, x: torch.Tensor, update_grid = False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation = 1.0, regularize_entropy = 1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )

## (128, )

In [9]:
# class FakeNewsDetectionModel(nn.Module):
#     def __init__(self, input_size, hidden_units, output_size):
#         super(FakeNewsDetectionModel, self).__init__()
#         self.ltc_cell = LiquidNet(hidden_units).to("cuda" if torch.cuda.is_available() else "cpu")  # LiquidNet cell
#         self.kan_1 = KANLinear(hidden_units, hidden_units // 2, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu") # KAN
#         self.kan_2 = KANLinear(hidden_units // 2, 1, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu")

#     def forward(self, x):
#         # LiquidNet requires an initial hidden state
#         batch_size = x.size(0)
#         hidden_state = torch.zeros(batch_size, self.ltc_cell._num_units).to(x.device)

#         # Pass input through the LiquidNet cell
#         outputs, _ = self.ltc_cell(x, hidden_state)

#         # Pass LiquidNet output to KAN
#         outputs = self.kan_1(outputs)
#         outputs = self.kan_2(outputs)

#         return outputs

In [10]:
# class FakeNewsDetectionModel(nn.Module):
#     def __init__(self, input_size, hidden_units, output_size):
#         super(FakeNewsDetectionModel, self).__init__()
#         self.l1 = nn.Linear(input_size, hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.l2 = nn.Linear(hidden_units // 8, output_size).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.ltc_cell = LiquidNet(hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")  # LiquidNet cell
#         self.kan_1 = KANLinear(hidden_units // 2, hidden_units // 4, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu") # KAN
#         self.kan_2 = KANLinear(hidden_units // 4, hidden_units // 8, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu")

#     def forward(self, x):
#         # LiquidNet requires an initial hidden state
#         batch_size = x.size(0)
#         hidden_state = torch.zeros(batch_size, self.ltc_cell._num_units).to(x.device)
        
#         outputs = self.l1(x)
#         # Pass input through the LiquidNet cell
#         outputs, _ = self.ltc_cell(outputs, hidden_state)

#         # Pass LiquidNet output to KAN
#         outputs = self.kan_1(outputs)
#         outputs = self.kan_2(outputs)
        
#         outputs = self.l2(outputs)

#         return outputs

## (256, )

In [11]:
# class FakeNewsDetectionModel(nn.Module):
#     def __init__(self, input_size, hidden_units, output_size):
#         super(FakeNewsDetectionModel, self).__init__()
#         self.l2 = nn.Linear(hidden_units, hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.l1 = nn.Linear(input_size, hidden_units * 8).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.gru_1 = nn.GRU(hidden_units * 2, hidden_units // 2, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.ltc_cell = LiquidNet(hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")  # LiquidNet cell
#         self.conv_1 = nn.Conv1d(hidden_units * 8, hidden_units * 2, kernel_size = 1).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.kan_1 = KANLinear(hidden_units // 2, hidden_units // 4, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu") # KAN
#         self.kan_2 = KANLinear(hidden_units // 4, output_size, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu")

#     def forward(self, x):
#         # LiquidNet requires an initial hidden state
#         batch_size = x.size(0)
#         hidden_state = torch.zeros(batch_size, self.ltc_cell._num_units).to(x.device)
        
#         outputs = self.l1(x)
#         outputs = self.conv_1(outputs.reshape(batch_size, outputs.shape[1], 1))
#         outputs, _ = self.gru_1(outputs.reshape(batch_size, 1, outputs.shape[1]))
#         outputs = self.l2(outputs.reshape(batch_size, -1))
#         # Pass input through the LiquidNet cell
#         outputs, _ = self.ltc_cell(outputs, hidden_state)

#         # Pass LiquidNet output to KAN
#         outputs = self.kan_1(outputs)
#         outputs = self.kan_2(outputs)

#         return outputs

## (512, )

In [12]:
# class FakeNewsDetectionModel(nn.Module):
#     def __init__(self, input_size, hidden_units, output_size):
#         super(FakeNewsDetectionModel, self).__init__()
#         self.l2 = nn.Linear(hidden_units, hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.l1 = nn.Linear(input_size, hidden_units * 8).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.gru_1 = nn.GRU(hidden_units * 2, hidden_units // 2, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.ltc_cell = LiquidNet(hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")  # LiquidNet cell
#         self.conv_1 = nn.Conv1d(hidden_units * 8, hidden_units * 2, kernel_size = 1).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.kan_1 = KANLinear(hidden_units // 2, hidden_units // 4, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu") # KAN
#         self.kan_2 = KANLinear(hidden_units // 4, output_size, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu")

#     def forward(self, x):
#         # LiquidNet requires an initial hidden state
#         batch_size = x.size(0)
#         hidden_state = torch.zeros(batch_size, self.ltc_cell._num_units).to(x.device)
        
#         outputs = self.l1(x)
#         outputs = self.conv_1(outputs.reshape(batch_size, outputs.shape[1], 1))
#         outputs, _ = self.gru_1(outputs.reshape(batch_size, 1, outputs.shape[1]))
#         outputs = self.l2(outputs.reshape(batch_size, -1))
#         # Pass input through the LiquidNet cell
#         outputs, _ = self.ltc_cell(outputs, hidden_state)

#         # Pass LiquidNet output to KAN
#         outputs = self.kan_1(outputs)
#         outputs = self.kan_2(outputs)

#         return outputs

In [19]:
# class FakeNewsDetectionModel(nn.Module):
#     def __init__(self, input_size, hidden_units, output_size):
#         super(FakeNewsDetectionModel, self).__init__()
#         self.l2 = nn.Linear(hidden_units, hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.l1 = nn.Linear(input_size, hidden_units * 8).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.l3 = nn.Linear(hidden_units // 2, hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.gru_1 = nn.GRU(hidden_units * 2, hidden_units // 2, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.ltc_cell = LiquidNet(hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")  # LiquidNet cell
#         self.conv_1 = nn.Conv1d(hidden_units * 8, hidden_units * 2, kernel_size = 1).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.kan_1 = KANLinear(hidden_units // 2, hidden_units // 4, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu") # KAN
#         self.kan_2 = KANLinear(hidden_units // 4, output_size, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu")

#     def forward(self, x):
#         # LiquidNet requires an initial hidden state
#         batch_size = x.size(0)
#         hidden_state = torch.zeros(batch_size, self.ltc_cell._num_units).to(x.device)
        
#         outputs = self.l1(x)
#         outputs = self.conv_1(outputs.reshape(batch_size, outputs.shape[1], 1))
#         outputs, _ = self.gru_1(outputs.reshape(batch_size, 1, outputs.shape[1]))
#         outputs = self.l2(outputs.reshape(batch_size, -1))
#         # Pass input through the LiquidNet cell
#         outputs, _ = self.ltc_cell(outputs, hidden_state)
#         outputs = self.l3(outputs)

#         # Pass LiquidNet output to KAN
#         outputs = self.kan_1(outputs)
#         outputs = self.kan_2(outputs)

#         return outputs

## (120, 128, )

In [35]:
# class FakeNewsDetectionModel(nn.Module):
#     def __init__(self, input_size, hidden_units, output_size):
#         super(FakeNewsDetectionModel, self).__init__()
#         self.multihead_attn = nn.MultiheadAttention(input_size, 4, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.gru_1 = nn.GRU(input_size, hidden_units // 4, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.gru_2 = nn.GRU(hidden_units // 2, hidden_units // 16, num_layers = 1, bidirectional = False, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.fc = nn.Linear(120 * hidden_units // 16, hidden_units * 4).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.ltc_cell = LiquidNet(hidden_units * 2).to("cuda" if torch.cuda.is_available() else "cpu")  # LiquidNet cell
#         self.kan_1 = KANLinear(hidden_units * 2, hidden_units, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu") # KAN
#         self.kan_2 = KANLinear(hidden_units, hidden_units // 2, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu")
#         self.o = nn.Linear(hidden_units // 2, output_size).to("cuda" if torch.cuda.is_available() else "cpu")

#     def forward(self, x):
#         # LiquidNet requires an initial hidden state
#         batch_size = x.size(0)
#         hidden_state = torch.zeros(batch_size, self.ltc_cell._num_units).to(x.device)
        
#         outputs, _ = self.multihead_attn(x, x, x)
#         outputs, _ = self.gru_1(outputs)
#         outputs, _ = self.gru_2(outputs)
        
#         #Pass through KAN and LTC
#         outputs = self.fc(outputs.reshape(batch_size, 120 * hidden_units // 16))
#         outputs, _ = self.ltc_cell(outputs, hidden_state)
#         outputs = self.kan_1(outputs)
#         outputs = self.kan_2(outputs)
#         outputs = self.o(outputs)

#         return outputs

In [ ]:
class FakeNewsDetectionModel(nn.Module):
    def __init__(self, input_size, hidden_units, output_size):
        super(FakeNewsDetectionModel, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(input_size, 4, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.gru_1 = nn.GRU(input_size, hidden_units // 4, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.fc = nn.Linear(120 * hidden_units // 2, hidden_units * 4).to("cuda" if torch.cuda.is_available() else "cpu")
        self.ltc_cell = LiquidNet(hidden_units * 2).to("cuda" if torch.cuda.is_available() else "cpu")  # LiquidNet cell
        self.kan_1 = KANLinear(hidden_units * 2, hidden_units, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu") # KAN
        self.kan_2 = KANLinear(hidden_units, hidden_units // 2, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu")
        self.o = nn.Linear(hidden_units // 2, output_size).to("cuda" if torch.cuda.is_available() else "cpu")

    def forward(self, x):
        # LiquidNet requires an initial hidden state
        batch_size = x.size(0)
        hidden_state = torch.zeros(batch_size, self.ltc_cell._num_units).to(x.device)
        
        outputs, _ = self.multihead_attn(x, x, x)
        outputs, _ = self.gru_1(outputs)
        
        #Pass through KAN and LTC
        outputs = self.fc(outputs.reshape(batch_size, 120 * hidden_units // 2))
        outputs, _ = self.ltc_cell(outputs, hidden_state)
        outputs = self.kan_1(outputs)
        outputs = self.kan_2(outputs)
        outputs = self.o(outputs)

        return outputs

## Training

In [36]:
def train_validate(model, train_loader, val_loader, num_epochs = 10, learning_rate = 0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate, betas = (0.9, 0.999))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience = 5, factor = 0.9)
    train_accuracies = []
    val_accuracies = []

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        train_loss = 0.0
        y_true_train = []
        y_pred_train = []

        for word_embeddings, labels in tqdm(train_loader):
            word_embeddings, labels = word_embeddings.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(word_embeddings.float())  # Outputs should have shape [batch_size, num_classes]
            loss = criterion(outputs, labels.unsqueeze(1).float())  # CrossEntropyLoss expects raw logits and class indices
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            # Get predicted class indices
            outputs = nn.Sigmoid()(outputs)
            y_true_train.extend(labels.cpu().numpy())
            y_pred_train.extend((outputs > 0.5).detach().cpu().numpy())

        # Calculate accuracy using sklearn
        train_accuracy = accuracy_score(y_true_train, y_pred_train) * 100
        train_accuracies.append(train_accuracy)
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Learning Rate: {optimizer.param_groups[-1]['lr']:.5f}")

        # Validation Phase
        val_loss = 0.0
        y_true_val = []
        y_pred_val = []

        with torch.no_grad():
            for word_embeddings, labels in val_loader:
                word_embeddings, labels = word_embeddings.to(device), labels.to(device)

                outputs = model(word_embeddings.float())
                loss = criterion(outputs, labels.unsqueeze(1).float())
                val_loss += loss.item()

                # Get predicted class indices
                outputs = nn.Sigmoid()(outputs)
                y_true_val.extend(labels.cpu().numpy())
                y_pred_val.extend((outputs > 0.5).detach().cpu().numpy())

        scheduler.step(val_loss)

        # Calculate validation accuracy using sklearn
        val_accuracy = accuracy_score(y_true_val, y_pred_val) * 100
        val_accuracies.append(val_accuracy)
        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")
        
    return train_accuracies, val_accuracies

In [37]:
def test(model, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    test_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()
    y_true_test = []
    y_pred_test = []

    with torch.no_grad():
        for word_embeddings, labels in test_loader:
            word_embeddings, labels = word_embeddings.to(device), labels.to(device)

            outputs = model(word_embeddings.float())
            loss = criterion(outputs, labels.unsqueeze(1).float())
            test_loss += loss.item()

            # Get predicted class indices
            outputs = nn.Sigmoid()(outputs)
            y_true_test.extend(labels.cpu().numpy())
            y_pred_test.extend((outputs > 0.5).detach().cpu().numpy())

    test_accuracy = accuracy_score(y_true_test, y_pred_test) * 100
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

In [38]:
# Initialize the model
input_size = 128  # Input size matches your embeddings
hidden_units = 64  # Number of units for LiquidNet
output_size = 1  # Binary classification (real/fake)
model = FakeNewsDetectionModel(input_size, hidden_units, output_size)

In [39]:
# Provide the model, input size (excluding batch size), and device
summary(model, input_size = [(120, 128, )], device = "cuda" if torch.cuda.is_available() else "cpu")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
MultiheadAttention-1  [[-1, 120, 128], [-1, 120, 120]]               0
               GRU-2  [[-1, 120, 32], [-1, 2, 16]]               0
            Linear-3                  [-1, 256]         983,296
         LiquidNet-4     [[-1, 128], [-1, 128]]               0
              SiLU-5                  [-1, 128]               0
         KANLinear-6                   [-1, 64]               0
              SiLU-7                   [-1, 64]               0
         KANLinear-8                   [-1, 32]               0
            Linear-9                    [-1, 1]              33
Total params: 983,329
Trainable params: 983,329
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.06
Forward/backward pass size (MB): 1688.56
Params size (MB): 3.75
Estimated Total Size (MB): 1692.37
---------------------------

In [40]:
# Train and validate
train_accuracies, val_accuracies = train_validate(model, train_loader, val_loader, num_epochs = 25, learning_rate = 0.0001)

  0%|          | 0/2147 [00:00<?, ?it/s]

Epoch 1/25, Train Loss: 864.6322, Train Accuracy: 78.11%, Learning Rate: 0.00010
Validation Loss: 96.5669, Validation Accuracy: 80.74%


  0%|          | 0/2147 [00:00<?, ?it/s]

Epoch 2/25, Train Loss: 722.7929, Train Accuracy: 82.37%, Learning Rate: 0.00010
Validation Loss: 91.9800, Validation Accuracy: 81.97%


  0%|          | 0/2147 [00:00<?, ?it/s]

...

Epoch 23/25, Train Loss: 596.8246, Train Accuracy: 85.41%, Learning Rate: 0.00009
Validation Loss: 84.5945, Validation Accuracy: 83.81%


  0%|          | 0/2147 [00:00<?, ?it/s]

Epoch 24/25, Train Loss: 593.8223, Train Accuracy: 85.59%, Learning Rate: 0.00009
Validation Loss: 84.3089, Validation Accuracy: 83.35%


  0%|          | 0/2147 [00:00<?, ?it/s]

Epoch 25/25, Train Loss: 592.4695, Train Accuracy: 85.56%, Learning Rate: 0.00009
Validation Loss: 84.0812, Validation Accuracy: 83.73%


In [41]:
# No need to run, aleady trained and saved

np.save('./Train_Accuracies_M8.npy', np.array(train_accuracies))
np.save('./Validation_Accuracies_M8.npy', np.array(val_accuracies))

In [40]:
# Test the model
test(model, test_loader)

Test Loss: 142.1949, Test Accuracy: 74.48%
